# 在 Colab 运行本项目
本单元将：
- 挂载 Google Drive
- 指定项目目录并切换工作路径
- 安装除 PyTorch 之外的依赖（避免与 Colab 自带 PyTorch 冲突）
- 检查 CUDA/GPU 与 PyTorch
- 尝试运行 main.py --help

In [ ]:
!nvidia-smi

Sun Dec 14 04:52:11 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
# 在任何环境都可安全运行的 Colab 探测与挂载（带超时）
import os, sys, time

def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

print('IN_COLAB:', in_colab())
if in_colab():
    try:
        from google.colab import drive  # type: ignore
        # 如果已挂载则跳过
        if not os.path.exists('/content/drive'):
            # 挂载但设置超时提示，避免一直卡住
            print('Mounting Google Drive... (watch for the auth popup)')
            drive.mount('/content/drive', force_remount=False)
        else:
            print('Drive already mounted.')
    except Exception as e:
        print('Drive mount skipped:', e)
else:
    print('Not in Colab, skipping drive.mount.')

IN_COLAB: True
Mounting Google Drive... (watch for the auth popup)
Drive mount skipped: mount failed


In [2]:
# 在 Colab 自动克隆仓库并进入目录
import os, sys, subprocess
REPO_URL = 'https://github.com/Shen-An/transferattack.git'
TARGET_DIR = '/content/TransferAttack'
if not os.path.exists(TARGET_DIR):
    subprocess.check_call(['git', 'clone', REPO_URL, TARGET_DIR])
os.chdir(TARGET_DIR)
print('CWD:', os.getcwd())
!git rev-parse --short HEAD

In [ ]:
# 设置项目目录：优先使用 Drive:/MyDrive/TransferAttack，否则使用 /content/TransferAttack
import os
drive_path = '/content/drive/MyDrive/TransferAttack'
local_path = '/content/TransferAttack'
PROJECT_DIR = drive_path if os.path.exists(drive_path) else local_path
os.makedirs(PROJECT_DIR, exist_ok=True)
print('PROJECT_DIR =', PROJECT_DIR)

In [ ]:
# 切换到项目目录并浏览文件
import os
os.chdir(PROJECT_DIR)
print('CWD:', os.getcwd())
print('Files:', sorted(os.listdir())[:50])

In [ ]:
# 检查 PyTorch/CUDA
try:
    import torch, torchvision
    print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
    print('CUDA available:', torch.cuda.is_available(), 'device_count:', torch.cuda.device_count())
except Exception as e:
    print('PyTorch not available:', e)

In [ ]:
# 生成 Colab 专用依赖清单：排除 torch/torchvision 与各类 --index-url
import os, re
req_in = os.path.join(os.getcwd(), 'requirements.txt')
req_out = os.path.join(os.getcwd(), 'colab_requirements.txt')
keep = []
if os.path.exists(req_in):
    with open(req_in, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith('#'):
                continue
            if s.startswith('--index-url') or s.startswith('--extra-index-url'):
                continue
            if re.search(r'^(torch|torchvision)(\b|==|>=|<=|\+)', s):
                continue
            keep.append(s)
    with open(req_out, 'w', encoding='utf-8') as f:
        f.write(''.join(keep) + ('' if keep else ''))
    print('Wrote', req_out, 'with', len(keep), 'lines')
else:
    print('requirements.txt not found at', req_in)

In [ ]:
# 安装非 PyTorch 依赖
import os, sys, subprocess
req_out = os.path.join(os.getcwd(), 'colab_requirements.txt')
if os.path.exists(req_out) and os.path.getsize(req_out) > 0:
    code = subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', '-r', req_out])
    print('pip exit code:', code)
else:
    print('No extra dependencies to install; skipping.')

In [ ]:
# 可选：验证 GPU 信息（Colab 运行时需启用 GPU）
import os
os.system('nvidia-smi')

In [ ]:
# 尝试运行 main.py --help
import os, sys, subprocess
if os.path.exists('main.py'):
    ret = subprocess.call([sys.executable, 'main.py', '--help'])
    print('main.py --help exit code:', ret)
else:
    print('main.py not found in', os.getcwd())

In [ ]:
# 生成 CIFAR-10 到 ./data/images 并写 labels.csv
import os, csv
from torchvision import datasets, transforms
from PIL import Image

data_dir = "./data"
images_dir = os.path.join(data_dir, "images")
os.makedirs(images_dir, exist_ok=True)

transform = transforms.Compose([transforms.ToTensor()])
dataset = datasets.CIFAR10(root="./_cifar_cache", train=False, download=True, transform=transform)
print("CIFAR10 test size:", len(dataset))

labels_path = os.path.join(data_dir, "labels.csv")
with open(labels_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["filename", "label"])  # 脚本读取的是文件名，不含路径
    for idx in range(len(dataset)):
        img, label = dataset[idx]
        fn = f"img_{idx:05d}.png"
        fp = os.path.join(images_dir, fn)
        Image.fromarray((img.permute(1,2,0).numpy() * 255).astype("uint8")).save(fp)
        w.writerow([fn, label])
        if idx % 1000 == 0:
            print(f"saved {idx} images...")
print("完成：", images_dir, "和", labels_path)

In [ ]:
# GPU 状态检查与一次 nvidia-smi
import torch, os
print("torch.cuda.is_available():", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("current device:", torch.cuda.current_device())
    print("device name:", torch.cuda.get_device_name(torch.cuda.current_device()))
    os.system("nvidia-smi -L")
    os.system("nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader,nounits")

import sys, subprocess

cmd = [
    sys.executable,
    "main.py",
    "-e",
    "--attack", "fgsm",
    "--eps", "0.062745",         # 16/255
    "--alpha", "0.062745",       # FGSM 一步，alpha=eps
    "--batchsize", "32",
    "--model", "resnet50",
    "--GPU_ID", "0",
    "--input_dir", "./data",
    "--output_dir", "./results_fgsm"
]
print("Running:", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
ret = proc.wait()
print("Exit code:", ret)

# 结束后再看一次 GPU 利用率与输出目录
if torch.cuda.is_available():
    os.system("nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader,nounits")
out_dir = "./results_fgsm"
print("Output exists:", os.path.isdir(out_dir))
if os.path.isdir(out_dir):
    print("Files:", sorted(os.listdir(out_dir))[:20])

In [ ]:
# 用 CIFAR-10 预训练模型评估原/对抗的准确率与攻击成功率
# filepath: e:\TransferAttack\TransferAttack\1.ipynb
import os, pandas as pd, torch, torchvision.transforms as T
from PIL import Image
from torchvision import datasets

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.hub.load('chenyaofo/pytorch-cifar-models', 'cifar10_resnet20', pretrained=True).to(device)
model.eval()

cifar_classes = datasets.CIFAR10(root="./_cifar_cache", train=False, download=False).classes
transform = T.Compose([
    T.Resize((32, 32)),
    T.ToTensor(),
    T.Normalize(mean=[0.4914, 0.4822, 0.4465],
                std=[0.2023, 0.1994, 0.2010])
])

labels_df = pd.read_csv("./data/labels.csv")
f2l = dict(zip(labels_df["filename"], labels_df["label"]))

def predict_id(fp):
    x = transform(Image.open(fp).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        pred = torch.argmax(logits, dim=1).item()
    return pred

src_dir = "./data/images"
adv_dir = "./results_fgsm"
files = sorted(os.listdir(adv_dir))

orig_correct = 0
adv_correct = 0
success = 0  # 攻击成功=原图预测正确且对抗图预测错误，或两者类别不同（按需选择）
for fn in files:
    gt = f2l.get(fn, None)
    if gt is None: 
        continue
    orig_pred = predict_id(os.path.join(src_dir, fn))
    adv_pred  = predict_id(os.path.join(adv_dir, fn))
    orig_correct += int(orig_pred == gt)
    adv_correct  += int(adv_pred == gt)
    success      += int(orig_pred != adv_pred)  # 迁移改变了预测

n = len(files)
print(f"Samples: {n}")
print(f"Original accuracy: {orig_correct/n:.3f}")
print(f"Adversarial accuracy: {adv_correct/n:.3f}")
print(f"Attack success rate (prediction changed): {success/n:.3f}")

In [ ]:
# 原/对抗并排同尺寸显示，并输出Top-1置信度（CIFAR-10）
import os
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torchvision.transforms as T
from torchvision import datasets

# 目录与标签
src_dir = "./data/images"
adv_dir = "./results_fgsm"
labels_df = pd.read_csv("./data/labels.csv")
f2l = dict(zip(labels_df["filename"], labels_df["label"]))
cifar_classes = datasets.CIFAR10(root="./_cifar_cache", train=False, download=False).classes

# 设备与模型（CIFAR-10预训练）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.hub.load('chenyaofo/pytorch-cifar-models', 'cifar10_resnet20', pretrained=True).to(device)
model.eval()

# CIFAR-10预处理
transform = T.Compose([
    T.Resize((32, 32)),
    T.ToTensor(),
    T.Normalize(mean=[0.4914, 0.4822, 0.4465],
                std=[0.2023, 0.1994, 0.2010])
])

def top1(fp):
    img = Image.open(fp).convert("RGB")
    x = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0]
    conf, idx = torch.max(probs, dim=0)
    return cifar_classes[int(idx)], float(conf)

# 选择文件并布局
files = sorted(os.listdir(adv_dir))[:8]
rows, cols = len(files), 2

plt.figure(figsize=(cols*4, rows*3))
for i, fn in enumerate(files):
    orig_fp = os.path.join(src_dir, fn)
    adv_fp  = os.path.join(adv_dir, fn)

    # 真实标签
    label_id = f2l.get(fn, None)
    label_name = cifar_classes[label_id] if label_id is not None else "NA"

    # 预测Top-1与置信度
    orig_pred, orig_conf = top1(orig_fp)
    adv_pred,  adv_conf  = top1(adv_fp)

    # 左：原图
    ax1 = plt.subplot(rows, cols, i*2 + 1)
    ax1.imshow(Image.open(orig_fp).convert("RGB"))
    ax1.set_title(f"{fn}\nGT={label_name} | Pred={orig_pred} ({orig_conf:.2f})")
    ax1.axis("off")

    # 右：对抗图
    ax2 = plt.subplot(rows, cols, i*2 + 2)
    ax2.imshow(Image.open(adv_fp).convert("RGB"))
    ax2.set_title(f"Adversarial\nPred={adv_pred} ({adv_conf:.2f})")
    ax2.axis("off")

plt.tight_layout()
plt.show()